# 03 — Reading the Spark UI

A whirlwind tour of the live Spark UI and the History Server. We first kick off a small job so there's something to look at.

In [ ]:
from spark_session import get_spark
spark = get_spark("03-ui-tour")

# A toy job: 5M-row range, grouped aggregate. Heavy enough to populate stages but finishes in seconds.
(spark.range(5_000_000)
      .selectExpr("id", "id % 13 AS bucket")
      .groupBy("bucket").count()
      .orderBy("bucket")
      .show())

## Live UI — http://localhost:8080

Open the master UI in another tab. You'll see:

- **Workers** — one row, with cores/memory the worker reports.
- **Running / Completed Applications** — `03-ui-tour` shows up here. Click its name to drill in.
- Inside the application:
  - **Jobs** — each call to an action (`.show()`, `.count()`) becomes a job.
  - **Stages** — DAG of shuffle boundaries. Look for skew (a single straggler task tells you a partition is hot).
  - **Executors** — per-executor task counts and shuffle bytes. Useful for spotting GC pressure.
  - **SQL / DataFrame** — every action above is rendered as a physical plan with row counts at each operator.

## Worker UI — http://localhost:8081

Shows the worker process's view: the executors it launched, their logs (stdout/stderr links), and resource usage.

## History Server — http://localhost:18080

Once a Spark application *finishes*, its event log is uploaded to `s3a://spark-logs/events/` and the History Server picks it up within ~15s. Use this UI to investigate jobs that have already exited (the live UI dies with the driver).

In [ ]:
# Stop the session so the History Server can ingest the completed event log.
spark.stop()